# Stage 2: fine-tune the ParaDetox T5 on GYAFC (informal → formal)
This is the second stage of a two-stage curriculum:

| Stage | Notebook | Starts from | Data | Task prefix |
|---|---|---|---|---|
| 1 | `train_detox_stage1.ipynb` | `google-t5/t5-small` | ParaDetox (toxic → neutral) | `detoxify: ` |
| 2 | **this notebook** | the stage-1 model | GYAFC (informal → formal) | `formalize: ` |

Stage 1 must be finished first. This notebook loads the exported stage-1 model with `from_pretrained` and trains it on GYAFC with a fresh optimizer. It does **not** use the Trainer's `resume_from_checkpoint`, which restores the old optimizer and scheduler state and is meant for continuing an interrupted run of the same job ([Trainer docs](https://huggingface.co/docs/transformers/main_classes/trainer)).

Training on a second task can erase part of the first (catastrophic forgetting). Section 11 runs the same `detoxify:` prompts before and after this stage so you can see how much detoxification survived.

**In Colab:** choose **Runtime → Change runtime type → GPU**, then run the cells in order. Upload the stage-1 model ZIP and `GYAFC_Corpus.zip` to `/content` first.

Set `SMOKE_TEST = True` in the configuration cell for a quick end-to-end check before full training. A smoke-test model is only a debugging artifact, not a useful trained model.

GYAFC is distributed under a Yahoo/Grammarly research license. Do not commit the corpus to this repository, and do not redistribute it.

Sources: [GYAFC (Rao & Tetreault, 2018)](https://aclanthology.org/N18-1012/), [ParaDetox](https://huggingface.co/datasets/s-nlp/paradetox), [T5](https://huggingface.co/docs/transformers/model_doc/t5), and [Hugging Face sequence-to-sequence training](https://huggingface.co/docs/transformers/tasks/translation).

## 1. Install dependencies
Use a fresh Colab runtime with Python 3.10 or newer. Colab provides PyTorch; install the remaining packages below. If Colab asks to restart the session, restart it and continue from the next cell. These versions use `eval_strategy` and `processing_class` in the Trainer API.

`sacrebleu` is used only by the optional BLEU cell in section 12.

In [ ]:
%pip install -q "transformers==4.57.3" "datasets==4.4.1" "accelerate==1.12.0" "sentencepiece==0.2.1" "sacrebleu==2.4.3"

## 2. Configure the run
These are the main settings to edit. The effective training batch size on one GPU is `8 × 2 = 16`. Full precision is used for both training and evaluation. All model parameters will be trained.

**Stage-1 model.** `STAGE1_MODEL` is the folder name (or path) of the exported ParaDetox model; a ZIP with the same name is extracted automatically. Run `train_detox_stage1.ipynb` to produce it, or change `STAGE1_MODEL` to match whatever you exported. `STAGE1_PREFIX` must be the prefix stage 1 was trained with.

**Learning rate.** The Hugging Face T5 docs note that `1e-4` and `3e-4` work well for most tasks. This stage uses the lower value: the model is already fine-tuned, and a smaller step size changes it less, which should reduce forgetting of the detox task. `train_formal.ipynb` uses `3e-4` from plain `t5-small`, so the two runs differ in both starting point and learning rate.

GYAFC ships its own train/tune/test splits, so this notebook uses them directly rather than resampling. `tune` becomes the validation split.

The `tune` and `test` folders are **cross-named**: the source sentences are in `informal`, and their four human formal references are `formal.ref0`–`formal.ref3`. (`informal.ref0`–`informal.ref3` are references for the reverse formal→informal task and are ignored here.)

`MAX_LENGTH = 64` covers every validation and test sentence — the longest is 58 tokens — and all but 20 of the 104,562 training sources. Stage 1 used 128 because ParaDetox comments are longer; T5's relative position embeddings make the change harmless.

Smoke mode uses at most 128 training rows, 32 validation rows, 32 test rows, and 5 optimizer steps. Its output folders are separate from the full model.

In [ ]:
import os
import shutil
import zipfile
from collections import Counter
from pathlib import Path

import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

STAGE1_MODEL = "t5-small-paradetox"  # Exported folder (or its .zip) from train_detox_stage1.ipynb.
STAGE1_PREFIX = "detoxify: "          # The prefix stage 1 was trained with.
DATASET_NAME = "GYAFC"  # Local line-aligned corpus; not available on the Hugging Face Hub.
DOMAINS = ("Entertainment_Music", "Family_Relationships")
SPLIT_DIRS = {"train": "train", "validation": "tune", "test": "test"}
SOURCE_COLUMN = "informal"
TARGET_COLUMN = "formal"
REFERENCE_COUNT = 4          # formal.ref0 .. formal.ref3 in tune/ and test/
LABEL_REFERENCE_INDEX = 0    # Which human reference supplies the eval/test loss label.
DROP_TRAIN_OVERLAP = True    # Remove the few tune/test sources that also appear in train.
DROP_IDENTITY_PAIRS = False  # Keep train pairs whose formal side equals the informal side.
PREFIX = "formalize: "
GYAFC_DIRNAME = "GYAFC_Corpus"
EXTRACT_ROOT = Path("/content") if Path("/content").is_dir() else Path(".")
SEED = 42
MAX_LENGTH = 64
EPOCHS = 3
LEARNING_RATE = 1e-4
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
SMOKE_TEST = False
SMOKE_STEPS = 5

RUN_DIR = "./detox-formalize-t5-smoke-checkpoints" if SMOKE_TEST else "./detox-formalize-t5-checkpoints"
EXPORT_DIR = "./detox-formalize-t5-smoke" if SMOKE_TEST else "./detox-formalize-t5"
RESULTS_DIR = f"{EXPORT_DIR}-results"  # Just the metric JSONs, so they download separately.
os.environ["WANDB_DISABLED"] = "true"
set_seed(SEED)
print("CUDA GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
print("Mode:", "smoke test" if SMOKE_TEST else "full training")
if not torch.cuda.is_available():
    print("For Colab training, switch to a GPU runtime. CPU training will be much slower.")

## 3. Load the stage-1 ParaDetox model
The cell looks for a folder named `STAGE1_MODEL` (or uses it directly if it is a path), then for `STAGE1_MODEL.zip`, in the working directory, `/content`, and `models/`. It checks that the folder holds a T5 config and weights before loading anything.

Both the model and the tokenizer come from the stage-1 folder. The tokenizer is unchanged from `t5-small`, but loading it from the same folder keeps the pair consistent.

Before training, the cell rewrites a few toxic messages with the `detoxify:` prefix. Section 11 repeats these after training to show what the GYAFC stage did to the detox behavior.

In [ ]:
import json

STAGE1_SEARCH_DIRS = (Path("."), Path("/content"), Path("models"), Path("..") / "models")


def looks_like_model(path):
    weights = ("model.safetensors", "pytorch_model.bin")
    return (path / "config.json").is_file() and any((path / name).is_file() for name in weights)


def extract_model(archive_path):
    """Extract a ZIP whose top-level folder is STAGE1_MODEL, rejecting escaping paths."""
    root = archive_path.parent.resolve()
    target = (root / Path(STAGE1_MODEL).name).resolve()
    with zipfile.ZipFile(archive_path) as archive:
        members = [m for m in archive.infolist() if not m.filename.startswith("__MACOSX/")]
        for member in members:
            destination = (root / member.filename).resolve()
            if destination != target and not destination.is_relative_to(target):
                raise ValueError(f"Unexpected ZIP member outside {target.name}/: {member.filename}")
        archive.extractall(root, members=members)
    return target


def resolve_stage1():
    direct = Path(STAGE1_MODEL)
    if looks_like_model(direct):
        return direct.resolve()
    for directory in STAGE1_SEARCH_DIRS:
        candidate = directory / STAGE1_MODEL
        if looks_like_model(candidate):
            return candidate.resolve()
    for directory in STAGE1_SEARCH_DIRS:
        archive = directory / f"{STAGE1_MODEL}.zip"
        if archive.is_file():
            print("Extracting", archive.resolve())
            extracted = extract_model(archive)
            if looks_like_model(extracted):
                return extracted
    searched = "\n  ".join(str(directory.resolve()) for directory in STAGE1_SEARCH_DIRS)
    raise FileNotFoundError(
        f"Could not find the stage-1 model {STAGE1_MODEL}/ or {STAGE1_MODEL}.zip in:\n  {searched}\n"
        "Run train_detox_stage1.ipynb first, then upload its ZIP here or set STAGE1_MODEL to its path."
    )


STAGE1_DIR = resolve_stage1()
model_type = json.loads((STAGE1_DIR / "config.json").read_text())["model_type"]
assert model_type == "t5", f"Expected a T5 model in {STAGE1_DIR}, found {model_type!r}"

tokenizer = AutoTokenizer.from_pretrained(STAGE1_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(STAGE1_DIR)
assert all(parameter.requires_grad for parameter in model.parameters())
print("Stage-1 model:", STAGE1_DIR)


@torch.inference_mode()
def rewrite(generator, generator_tokenizer, prefix, text, max_length=MAX_LENGTH):
    inputs = generator_tokenizer(
        prefix + text, return_tensors="pt", max_length=max_length, truncation=True
    ).to(generator.device)
    output_ids = generator.generate(**inputs, max_new_tokens=max_length, num_beams=4, do_sample=False)
    return generator_tokenizer.decode(output_ids[0], skip_special_tokens=True)


DETOX_PROBES = [
    "Seriously, Maya, this invoice is a damn mess. Change the total from $480 to $450 before you send it.",
    "You dumped 18 broken laptops on my desk like an idiot. Get them out of room 204 by 3 PM.",
    "I'm sick of your bullshit excuses, Daniel. The client needs those 6 drawings by Friday.",
    "Did you even test this? This stupid retry loop hits the API 20 times. Replace it before merging.",
]
model.eval()
detox_before = [rewrite(model, tokenizer, STAGE1_PREFIX, text, max_length=128) for text in DETOX_PROBES]
for text, output in zip(DETOX_PROBES, detox_before):
    print("INPUT: ", text)
    print("STAGE 1:", output)
    print()

## 4. Provide the GYAFC corpus
GYAFC is license-restricted and is not on the Hugging Face Hub, so `load_dataset("name")` cannot fetch it. **Upload `GYAFC_Corpus.zip` to the Colab session at `/content` before running the cell below.** The cell searches the working directory and `/content` first, extracts the archive, and never prompts for an upload; if it finds nothing it lists every path it checked.

`/content` is wiped when the runtime recycles, so re-upload the archive after a restart. It is about 6.5 MB.

Extraction rejects absolute paths, members that escape the target folder, and symlinks, and it skips the `__MACOSX/` resource forks that the official archive contains.

In [ ]:
SEARCH_DIRS = (
    Path("."),            # The Colab working directory is /content.
    Path("/content"),
    Path("/content/data"),
    Path("data"),
    Path("..") / "data",
    Path("GYAFC_Corpus"),          # Repo root, when the archive was extracted there.
    Path("..") / "GYAFC_Corpus",   # Same, when the kernel runs from notebooks/.
)

def looks_like_corpus(path):
    """True only if the expected data files are present, not just the folder."""
    return all((path / domain / "train" / "informal").is_file() for domain in DOMAINS)


def safe_members(archive, target):
    """Yield only GYAFC_Corpus/* members, rejecting escaping paths and symlinks."""
    for member in archive.infolist():
        name = member.filename
        if name.startswith("__MACOSX/") or Path(name).name.startswith("._"):
            continue  # macOS resource forks in the official archive.
        if name != f"{GYAFC_DIRNAME}/" and not name.startswith(f"{GYAFC_DIRNAME}/"):
            raise ValueError(f"Unexpected ZIP member outside {GYAFC_DIRNAME}/: {name}")
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError(f"Symlink in archive: {name}")
        destination = (EXTRACT_ROOT / name).resolve()
        if destination != target and not destination.is_relative_to(target):
            raise ValueError(f"Unexpected ZIP member path: {name}")
        yield member


def extract_corpus(archive_path):
    target = (EXTRACT_ROOT / GYAFC_DIRNAME).resolve()
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(EXTRACT_ROOT, members=list(safe_members(archive, target)))
    return target


def resolve_corpus():
    for directory in SEARCH_DIRS:
        candidate = directory / GYAFC_DIRNAME
        if looks_like_corpus(candidate):
            return candidate.resolve()
    for directory in SEARCH_DIRS:
        archive = directory / f"{GYAFC_DIRNAME}.zip"
        if archive.is_file():
            print("Extracting", archive.resolve())
            extracted = extract_corpus(archive)
            if looks_like_corpus(extracted):
                return extracted
    searched = "\n  ".join(str(directory.resolve()) for directory in SEARCH_DIRS)
    raise FileNotFoundError(
        f"Could not find {GYAFC_DIRNAME}/ or {GYAFC_DIRNAME}.zip in:\n  {searched}\n"
        f"Upload {GYAFC_DIRNAME}.zip to /content, then re-run this cell."
    )


GYAFC_ROOT = resolve_corpus()
print("Corpus root:", GYAFC_ROOT)
for domain in DOMAINS:
    for directory in SPLIT_DIRS.values():
        assert (GYAFC_ROOT / domain / directory).is_dir(), f"Missing {domain}/{directory}"
print("Found both domains with train/tune/test.")

## 5. Load the parallel line files
Each GYAFC split is a set of plain UTF-8, line-aligned text files with no extensions, so the cell below reads them directly and builds a `DatasetDict` with `Dataset.from_dict`.

Both domains are concatenated, with a `domain` column so counts can be reported per domain. Line counts are asserted equal across every file in a split **before** any row is built — a line-aligned corpus fails silently otherwise, pairing each sentence with the wrong rewrite. Rows are dropped only when the source or any reference is blank after stripping, and always as a whole row so alignment is preserved.

Train has one crowdsourced formal rewrite per sentence. Validation and test keep all four human references in a `references` column, and reference `LABEL_REFERENCE_INDEX` supplies the `formal` column that `eval_loss` and `test_loss` score against. The four references are equally valid rewrites in different styles; index 0 is a fixed arbitrary choice, not the best one.

In [ ]:
def read_lines(path):
    text = Path(path).read_text(encoding="utf-8")
    lines = text.splitlines()
    # Every GYAFC file ends in a newline and has no interior blank lines. splitlines()
    # also breaks on \x0b \x0c \x85 \u2028 \u2029, so cross-check against newlines.
    assert len(lines) == text.count("\n"), f"{path}: unexpected line breaks or no final newline"
    return lines


def read_domain_split(domain, directory):
    base = GYAFC_ROOT / domain / directory
    sources = read_lines(base / SOURCE_COLUMN)
    if directory == "train":
        reference_files = [TARGET_COLUMN]
    else:
        # tune/ and test/ are cross-named: informal is the source, formal.ref* its references.
        reference_files = [f"{TARGET_COLUMN}.ref{index}" for index in range(REFERENCE_COUNT)]
    references = [read_lines(base / name) for name in reference_files]
    for name, column in zip(reference_files, references):
        assert len(column) == len(sources), (
            f"{domain}/{directory}: {name} has {len(column):,} lines, "
            f"but {SOURCE_COLUMN} has {len(sources):,}"
        )
    label_index = 0 if directory == "train" else LABEL_REFERENCE_INDEX
    rows, dropped = [], 0
    for position, source in enumerate(sources):
        source = source.strip()
        candidates = [column[position].strip() for column in references]
        if not source or not all(candidates):
            dropped += 1
            continue
        rows.append({
            SOURCE_COLUMN: source,
            TARGET_COLUMN: candidates[label_index],
            "references": candidates,
            "domain": domain,
        })
    print(f"{domain}/{directory}: {len(rows):,} rows from {len(sources):,} lines ({dropped} dropped as blank).")
    return rows


split_rows = {
    name: [row for domain in DOMAINS for row in read_domain_split(domain, directory)]
    for name, directory in SPLIT_DIRS.items()
}


def input_key(text):
    return " ".join(text.split()).casefold()


train_keys = {input_key(row[SOURCE_COLUMN]) for row in split_rows["train"]}
for name in ("validation", "test"):
    overlapping = sum(1 for row in split_rows[name] if input_key(row[SOURCE_COLUMN]) in train_keys)
    print(f"{name}: {overlapping} of {len(split_rows[name]):,} sources also appear in train.")
    if DROP_TRAIN_OVERLAP:
        split_rows[name] = [
            row for row in split_rows[name] if input_key(row[SOURCE_COLUMN]) not in train_keys
        ]

identity = sum(
    1 for row in split_rows["train"]
    if input_key(row[SOURCE_COLUMN]) == input_key(row[TARGET_COLUMN])
)
print(f"train: {identity:,} pairs whose formal side matches the informal side after normalization.")
if DROP_IDENTITY_PAIRS:
    split_rows["train"] = [
        row for row in split_rows["train"]
        if input_key(row[SOURCE_COLUMN]) != input_key(row[TARGET_COLUMN])
    ]


def to_dataset(rows):
    assert rows, "A split ended up empty."
    return Dataset.from_dict({column: [row[column] for row in rows] for column in rows[0]})


splits = DatasetDict({name: to_dataset(rows) for name, rows in split_rows.items()})
if SMOKE_TEST:
    limits = {"train": 128, "validation": 32, "test": 32}
    splits = DatasetDict({
        name: data.shuffle(seed=SEED).select(range(min(len(data), limits[name])))
        for name, data in splits.items()
    })
print(splits)
for name, data in splits.items():
    print(name, "domains:", dict(Counter(data["domain"])))
print("\ntrain example source:", splits["train"][0][SOURCE_COLUMN])
print("train example target:", splits["train"][0][TARGET_COLUMN])
test_example = splits["test"][0]
print("\ntest example source: ", test_example[SOURCE_COLUMN])
for index, reference in enumerate(test_example["references"]):
    print(f"test example ref{index}:  ", reference)
# Sanity check the cross-naming: these four must be formal rewrites of the source above.
assert test_example[TARGET_COLUMN] == test_example["references"][LABEL_REFERENCE_INDEX]

## 6. Tokenize inputs and targets
T5 learns a text-to-text task: `formalize: <informal sentence>` → `<formal rewrite>`. Use the same prefix at inference time — it is a learned task token, not one of T5's native pretraining prefixes, so the exported model is useless without the exact string. The stage-1 prefix `detoxify: ` stays meaningful to the model only as far as this stage does not overwrite it (see section 11).

Tokenize inputs and targets separately and truncate each to 64 tokens. Do not pad the whole dataset here. The data collator pads each batch to its longest sequence and uses `-100` for target padding, which the loss ignores.

The model and tokenizer were already loaded from the stage-1 folder in section 3.

In [ ]:
model.train()

def tokenize_batch(batch):
    inputs = [PREFIX + text for text in batch[SOURCE_COLUMN]]
    encoded = tokenizer(inputs, max_length=MAX_LENGTH, truncation=True)
    targets = tokenizer(text_target=batch[TARGET_COLUMN], max_length=MAX_LENGTH, truncation=True)
    encoded["labels"] = targets["input_ids"]
    return encoded

tokenized = splits.map(tokenize_batch, batched=True, remove_columns=splits["train"].column_names)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Tokenized fields:", tokenized["train"].column_names)

## 7. Train and keep the best checkpoint
For a full run, evaluate and save after every epoch. The lowest validation loss determines the best checkpoint, which is loaded automatically at the end. Keep at most two checkpoints to limit disk usage.

104,562 training rows at an effective batch of 16 give about 6,535 optimizer steps per epoch, so three epochs is roughly 19,605 steps and about 45 minutes on an A100 — against 4,935 steps and about 10 minutes for the ParaDetox run on a much smaller dataset.

In smoke mode, evaluate and save after the last of five steps instead. Checkpoints are separate from the final exported model. Re-running training starts a new run; use a new `RUN_DIR` if you want to preserve old checkpoints.

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=RUN_DIR,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    eval_strategy="steps" if SMOKE_TEST else "epoch",
    save_strategy="steps" if SMOKE_TEST else "epoch",
    eval_steps=SMOKE_STEPS if SMOKE_TEST else None,
    save_steps=SMOKE_STEPS if SMOKE_TEST else 500,
    max_steps=SMOKE_STEPS if SMOKE_TEST else -1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    logging_steps=1 if SMOKE_TEST else 50,
    fp16=False,
    bf16=False,
    optim="adamw_torch",
    report_to="none",
    push_to_hub=False,
    seed=SEED,
    data_seed=SEED,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=collator,
)
train_result = trainer.train()
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation loss:", trainer.state.best_metric)

## 8. Evaluate on held-out test data
Compute test loss only after checkpoint selection. This measures how well the model predicts one reference rewrite; it does **not** independently measure formality gain or meaning preservation. Section 12 scores against all four references, and the samples in section 10 are worth reading.

Two caveats on these losses. They score only reference `LABEL_REFERENCE_INDEX`, so never compare runs that differ in that setting. And training targets are a single noisy crowdsourced rewrite each, while validation and test references were curated — so training loss and validation loss are measured against different quality distributions and are less comparable than they were for ParaDetox.

This validation loss is not comparable to the stage-1 validation loss: it is a different dataset, task and label set.

The Trainer writes its metric JSONs into `RUN_DIR`, alongside gigabytes of checkpoints. This cell copies them into `RESULTS_DIR` so section 13 can archive them on their own, and prints the per-epoch curve so the numbers are readable even if no download succeeds.

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=tokenized["test"], metric_key_prefix="test")
print(test_metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_metrics("test", test_metrics)

# save_metrics writes into RUN_DIR, which also holds gigabytes of checkpoints.
# Copy just the JSONs somewhere small so they can be archived on their own.
results_dir = Path(RESULTS_DIR)
results_dir.mkdir(parents=True, exist_ok=True)
for name in ("all_results.json", "train_results.json", "test_results.json"):
    source = Path(RUN_DIR) / name
    if source.is_file():
        shutil.copy2(source, results_dir / name)

# trainer_state.json carries the full log history, including every epoch eval_loss.
best_checkpoint = trainer.state.best_model_checkpoint
state_file = Path(best_checkpoint) / "trainer_state.json" if best_checkpoint else Path(RUN_DIR) / "trainer_state.json"
if state_file.is_file():
    shutil.copy2(state_file, results_dir / "trainer_state.json")

print("Metrics folder:", results_dir.resolve())
print("Files:", sorted(path.name for path in results_dir.iterdir()))

# Print the curve too, so the numbers are readable without downloading anything.
print("Validation loss by epoch:")
for entry in trainer.state.log_history:
    if "eval_loss" in entry:
        print(f"  epoch {entry['epoch']:.0f}  eval_loss {entry['eval_loss']:.4f}")
print("Best:", trainer.state.best_metric, "at", best_checkpoint)
print("Test loss:", test_metrics.get("test_loss"))
if SMOKE_TEST:
    print("Smoke-test results only: these are not full-training quality measurements.")

## 9. Save and reload the model
Export the best model and its tokenizer. Reload from the saved directory to verify that inference works without the Trainer. The helper below handles one string at a time and returns an empty string for blank input. Longer inputs are truncated to the configured token limit.

In [ ]:
trainer.save_model(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

# Reload on CPU to avoid holding a second model copy in GPU memory.
# For faster inference in a separate session, move this model to "cuda".
inference_tokenizer = AutoTokenizer.from_pretrained(EXPORT_DIR)
inference_model = AutoModelForSeq2SeqLM.from_pretrained(EXPORT_DIR).to("cpu")
inference_model.eval()

@torch.inference_mode()
def formalize(text):
    if not isinstance(text, str):
        raise TypeError("text must be a string")
    if not text.strip():
        return ""
    inputs = inference_tokenizer(
        PREFIX + text, return_tensors="pt", max_length=MAX_LENGTH, truncation=True
    ).to(inference_model.device)
    output_ids = inference_model.generate(
        **inputs, max_new_tokens=MAX_LENGTH, num_beams=4, do_sample=False
    )
    return inference_tokenizer.decode(output_ids[0], skip_special_tokens=True)

assert formalize("   ") == ""
print("Saved and reloaded:", Path(EXPORT_DIR).resolve())

## 10. Inspect rewrites and try your own text
Compare input, references, and model output. Look for a more formal register and preservation of the original meaning. All four human references are shown, and they disagree with each other — that is the point, since there is no single correct formal rewrite. References and model outputs can both be imperfect. The first five test examples are a qualitative check, not a full quality evaluation.

In [ ]:
for example in splits["test"].select(range(min(5, len(splits["test"])))):
    print("INPUT: ", example[SOURCE_COLUMN])
    for index, reference in enumerate(example["references"]):
        print(f"REF{index}:  ", reference)
    print("MODEL: ", formalize(example[SOURCE_COLUMN]))
    print()

print("Your example:", formalize("gonna need that report asap, u totally dropped the ball on this one"))

## 11. Check what happened to detoxification
These are the same `detoxify:` prompts as in section 3, run through the stage-1 model before this training and through the exported stage-2 model now. If the stage-2 outputs have started formalizing instead of removing insults, or the insults come back, the GYAFC stage overwrote part of stage 1.

This is a four-example qualitative check, not a measurement. For a real comparison, run both models on `data/workplace_eval/input.jsonl` and score them with `secondthought evaluate --with-learned-metrics`.

In [ ]:
for text, before in zip(DETOX_PROBES, detox_before):
    print("INPUT:   ", text)
    print("STAGE 1: ", before)
    print("STAGE 2: ", rewrite(inference_model, inference_tokenizer, STAGE1_PREFIX, text, max_length=128))
    print("FORMAL:  ", formalize(text))
    print()

## 12. Multi-reference BLEU on the test split
GYAFC's standard metric is corpus BLEU against all four human references, so this cell generates test rewrites with the same decoding settings as `formalize()` and scores them with sacrebleu.

It is kept out of the Trainer deliberately: generating thousands of validation rewrites with beam search after every epoch would dominate the run, and validation loss already selects the checkpoint.

**Read the copy baseline first.** It is the BLEU of the unchanged informal source against the same four references. GYAFC references stay close to their sources, so simply echoing the input scores high. A model BLEU near the copy baseline means the model is barely editing, not that it is good.

BLEU here is pooled across both domains; the GYAFC paper reports them separately. Note also that the batched generator below pads its inputs while `formalize()` does not, so outputs can differ slightly — treat `formalize()` as the reference implementation for the exported model.

In [ ]:
import sacrebleu

GENERATION_BATCH_SIZE = 64

@torch.inference_mode()
def generate_all(texts):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    generator = inference_model.to(device)
    outputs = []
    for start in range(0, len(texts), GENERATION_BATCH_SIZE):
        chunk = [PREFIX + text for text in texts[start:start + GENERATION_BATCH_SIZE]]
        inputs = inference_tokenizer(
            chunk, return_tensors="pt", padding=True, max_length=MAX_LENGTH, truncation=True
        ).to(device)
        generated = generator.generate(
            **inputs, max_new_tokens=MAX_LENGTH, num_beams=4, do_sample=False
        )
        outputs += inference_tokenizer.batch_decode(generated, skip_special_tokens=True)
        print(f"  generated {min(start + GENERATION_BATCH_SIZE, len(texts)):,}/{len(texts):,}", flush=True)
    inference_model.to("cpu")
    return [text.strip() for text in outputs]

sources = [text.strip() for text in splits["test"][SOURCE_COLUMN]]
reference_streams = [
    [row[index] for row in splits["test"]["references"]] for index in range(REFERENCE_COUNT)
]
predictions = generate_all(sources)

model_bleu = sacrebleu.corpus_bleu(predictions, reference_streams)
copy_bleu = sacrebleu.corpus_bleu(sources, reference_streams)
unchanged = sum(1 for source, prediction in zip(sources, predictions) if source == prediction)

print(f"\nTest sentences:      {len(sources):,}")
print(f"Model BLEU (4 refs): {model_bleu.score:.2f}")
print(f"Copy BLEU (4 refs):  {copy_bleu.score:.2f}   <- echoing the input scores this")
print(f"BLEU over copy:      {model_bleu.score - copy_bleu.score:+.2f}")
print(f"Outputs identical to input: {unchanged:,} ({unchanged / len(sources):.1%})")
if SMOKE_TEST:
    print("Smoke-test results only: these are not full-training quality measurements.")

## 13. Download the results and the model
Two archives, smallest first. `RESULTS_DIR` is a few kilobytes and always transfers. `EXPORT_DIR` is about 225 MB, which is under the limit below.

**Do not archive `RUN_DIR`.** With `save_total_limit=2` it holds two checkpoints carrying optimizer state as well as weights, so it runs past a gigabyte. The VS Code Jupyter path tops out near 384 MB per file: the bytes arrive base64-encoded inside a JSON message and the extension buffers them as one JavaScript string, which caps at 2^29 bytes (`Cannot create a string longer than 0x1fffffe8 characters`). Colab's own browser file browser streams over HTTPS and has no such ceiling. Checkpoints are only useful for resuming training, so there is no reason to fetch them.

Keep `PREFIX = "formalize: "` and `MAX_LENGTH = 64` with the model, and note that it also still answers to `detoxify: ` to the extent section 11 showed: inference must use the same prefix string, and the model has no training signal for inputs longer than about 58 tokens.

In [ ]:
def archive_and_download(directory):
    """Zip one folder and hand it to the browser. Returns the archive path."""
    path = Path(directory).resolve()
    if not path.is_dir():
        print("Skipping", path, "- not found.")
        return None
    archive_path = shutil.make_archive(str(path), "zip", root_dir=path.parent, base_dir=path.name)
    size_mb = Path(archive_path).stat().st_size / 1_000_000
    print(f"{archive_path}  ({size_mb:,.1f} MB)")
    if size_mb > 384:
        print("  Over the ~384 MB transfer limit; fetch it from Colab's file browser instead.")
    try:
        from google.colab import files
    except ImportError:
        print("  Outside Colab: copy the ZIP from the path above.")
    else:
        files.download(archive_path)
    return archive_path

archive_and_download(RESULTS_DIR)   # Kilobytes: the metrics.
archive_and_download(EXPORT_DIR)    # ~225 MB: the model.